# SVO Partial Claim Structure: Where the Pipeline Succeeds and Fails

This notebook shows individual sentences from the corpus with their extracted
entities and verb structure. It covers:

1. Sentences where the pipeline correctly identifies a supported claim
2. Sentences where existential verbs (be/have) dominate — weak factual content
3. Single-entity sentences that inflate the denominator but cannot be verified
4. French-specific failures in Quebec — sentences where spaCy misidentifies
   the ROOT due to passive voice, nominalization, and modal constructions

In [1]:
import json, os, re, random
REPO = '/Users/caprice/MiscellaneousProjects/legislative-health-factuality'
os.makedirs(f'{REPO}/figures', exist_ok=True)

try:
    import spacy
    nlp_en = spacy.load('en_core_web_sm')
    print('English spaCy loaded')
except:
    nlp_en = None
    print('Missing en_core_web_sm — run: python -m spacy download en_core_web_sm')

try:
    import spacy
    nlp_fr = spacy.load('fr_core_news_sm')
    print('French spaCy loaded')
except:
    nlp_fr = None
    print('Missing fr_core_news_sm — run: python -m spacy download fr_core_news_sm')

English spaCy loaded
French spaCy loaded


In [2]:
print('Loading DB1 files...')
with open(f'{REPO}/data/dbpedia_pipeline/ontario/db1_claim_bearing.json') as f:
    ont_db1 = json.load(f)
with open(f'{REPO}/data/dbpedia_pipeline/quebec/db1_claim_bearing.json') as f:
    qc_db1 = json.load(f)
with open(f'{REPO}/data/dbpedia_pipeline/us/db1_claim_bearing.json') as f:
    us_db1 = json.load(f)
print(f'Ontario: {len(ont_db1):,} | Quebec: {len(qc_db1):,} | US: {len(us_db1):,}')

Loading DB1 files...
Ontario: 283,033 | Quebec: 334,522 | US: 376,347


In [3]:
def label(uri): return uri.split('/')[-1].replace('_',' ')

def parse(text, lang='en'):
    nlp = nlp_en if lang == 'en' else nlp_fr
    if not nlp: return {}
    doc = nlp(text)
    root = next((t for t in doc if t.dep_=='ROOT'), None)
    subjs = [t.text for t in doc if t.dep_ in ('nsubj','nsubjpass')]
    objs  = [t.text for t in doc if t.dep_ in ('dobj','obj','pobj')]
    return {'root': root.text if root else None,
            'pos': root.pos_ if root else None,
            'subj': subjs, 'obj': objs}

def show(s, lang='en'):
    print('-'*65)
    spkr = s.get('speaker_full', s.get('speaker',''))
    print(f"[{s.get('date','')}] {spkr} | {s.get('party','')}")
    print(f"Supported: {s.get('supported',False)} | Hedged: {s.get('hedged',False)}")
    print(f"Text: {s['text']}")
    ents = [label(u) for u in s.get('concept_uris',[])]
    print(f"Entities: {ents}")
    p = parse(s['text'], lang)
    if p:
        print(f"ROOT: '{p.get('root')}' (POS={p.get('pos')}) | Subj={p.get('subj')} | Obj={p.get('obj')}")

print('Helpers ready')

Helpers ready


In [4]:
# ── SECTION 1: WHERE IT WORKS ─────────────────────────────────────────────────
print('=== WHERE THE PIPELINE WORKS ===')
print('Supported sentences: 2+ entities, DBpedia confirms a relationship')
print()

good = [s for s in ont_db1
        if s.get('supported') and len(s.get('concept_uris',[])) >= 2
        and not s.get('hedged') and 40 < len(s['text']) < 200][:5]

for s in good:
    show(s, 'en')
    print()

=== WHERE THE PIPELINE WORKS ===
Supported sentences: 2+ entities, DBpedia confirms a relationship

-----------------------------------------------------------------
[1985-06-14] John Keith Riddell | Ontario Liberal Party
Supported: True | Hedged: False
Text: The minister is aware, I am sure, that it is very difficult for Canada to compete with the export subsidies of the United States and the European Community.
Entities: ['Canada', 'United States']
ROOT: 'am' (POS=AUX) | Subj=['minister', 'I', 'it', 'Canada'] | Obj=['subsidies', 'States']

-----------------------------------------------------------------
[1985-06-14] Monte Kwinter | Ontario Liberal Party
Supported: True | Hedged: False
Text: Miss Stephenson: I have a little difficulty with the mathematics of the member; obviously he needs a new computer or calculator.
Entities: ['Mathematics', 'Computer', 'Calculator']
ROOT: 'needs' (POS=VERB) | Subj=['I', 'he'] | Obj=['difficulty', 'mathematics', 'member', 'computer']

-------------

In [5]:
# ── SECTION 2: EXISTENTIAL VERBS ──────────────────────────────────────────────
print('=== EXISTENTIAL VERBS (be/have/do) ===')
print('Most common ROOT verbs — entities present but relationship is trivial')
print()

existential_roots = {'be','is','are','was','were','have','has','had','do','does','did'}

existential = []
for s in ont_db1:
    if (len(s.get('concept_uris',[])) >= 2
        and not s.get('supported')
        and 40 < len(s['text']) < 180):
        p = parse(s['text'], 'en')
        if p.get('root','').lower() in existential_roots:
            existential.append((s, p))
    if len(existential) >= 5: break

for s, p in existential:
    print('-'*65)
    print(f"Text: {s['text']}")
    print(f"ROOT: '{p['root']}' — existential, not an action verb")
    print(f"Entities: {[label(u) for u in s.get('concept_uris',[])[:3]]}")
    print()

=== EXISTENTIAL VERBS (be/have/do) ===
Most common ROOT verbs — entities present but relationship is trivial

-----------------------------------------------------------------
Text: Yesterday we had an opportunity to discuss the province's credit rating.
ROOT: 'had' — existential, not an action verb
Entities: ['Yesterday (Beatles song)', 'Credit rating']

-----------------------------------------------------------------
Text: It is important that we realize as much as one per cent of the people in North America are known to be carriers of hepatitis-B vir
ROOT: 'is' — existential, not an action verb
Entities: ['North America', 'Vir']

-----------------------------------------------------------------
Text: My question has to do with the deinstitutionalization of the mentally retarded.
ROOT: 'has' — existential, not an action verb
Entities: ['Deinstitutionalisation', 'Intellectual disability']

-----------------------------------------------------------------
Text: The Minister of Energy 

In [6]:
# ── SECTION 3: SINGLE ENTITY — UNVERIFIABLE ───────────────────────────────────
print('=== SINGLE ENTITY SENTENCES ===')
print('60.3% of Ontario DB1 — counted in denominator but can never be SPARQL-verified')
print()

single = [s for s in ont_db1
          if len(s.get('concept_uris',[])) == 1
          and 40 < len(s['text']) < 200][:5]

for s in single:
    print('-'*65)
    print(f"Text: {s['text']}")
    print(f"Single entity only: {label(s['concept_uris'][0])}")
    print(f"Cannot form SPARQL pair — unverifiable")
    print()

=== SINGLE ENTITY SENTENCES ===
60.3% of Ontario DB1 — counted in denominator but can never be SPARQL-verified

-----------------------------------------------------------------
Text: I would like to announce the implementation of the government's integrated homemaker program for Ontario's frail elderly and disabled adults.
Single entity only: Government of Ontario
Cannot form SPARQL pair — unverifiable

-----------------------------------------------------------------
Text: Over the past 15 years, the government of Ontario has been laying the foundati
Single entity only: Government of Ontario
Cannot form SPARQL pair — unverifiable

-----------------------------------------------------------------
Text: Speaker, on a point of privilege: The Minister of Community and Social Services gave a statement today about the integrated homemaker program.
Single entity only: Speaker of the Parliament of Sri Lanka
Cannot form SPARQL pair — unverifiable

---------------------------------------------

In [9]:
# ── SECTION 3b: SPOTLIGHT DISAMBIGUATION ERRORS ───────────────────────────────
print('=== SPOTLIGHT DISAMBIGUATION ERRORS ===')
print('Spotlight sometimes links common words to wrong DBpedia entities')
print()

bad_entities = {
    'Speaker of the Parliament of Sri Lanka': 'Should be: parliamentary procedure term',
    'Yesterday (Beatles song)': 'Should be: temporal adverb',
    'Vir': 'Should be: fragment from "virus"',
    'Setting Standards': 'Should be: gerund phrase, not an entity',
}

for bad, note in bad_entities.items():
    examples = [s for s in ont_db1
                if any(bad in uri for uri in s.get('concept_uris',[]))
                and 30 < len(s['text']) < 200][:1]
    for s in examples:
        print(f"  Entity linked: {bad}")
        print(f"  Note: {note}")
        print(f"  Text: {s['text'][:150]}")
        print()

=== SPOTLIGHT DISAMBIGUATION ERRORS ===
Spotlight sometimes links common words to wrong DBpedia entities

  Entity linked: Vir
  Note: Should be: fragment from "virus"
  Text: It is important that we realize as much as one per cent of the people in North America are known to be carriers of hepatitis-B vir



In [7]:
# ── SECTION 4: FRENCH FAILURES — FOR JOANIE ───────────────────────────────────
print('=== FRENCH GRAMMATICAL STRUCTURES THAT BREAK EXTRACTION ===')
print('Quebec sentences where spaCy ROOT is wrong — Joanie please annotate')
print()

fr_patterns = [
    ('Passive voice (a été + participle)',    r'a été|ont été|avait été|ont aussi été'),
    ('Modal + infinitive (pourrait/devrait)', r'pourrait |devrait |pourraient |devraient '),
    ('Nominalization (X-tion)',               r'la \w+tion|l\'\w+tion|les \w+tions|la \w+ment'),
    ('Impersonal (il faut/il s\'agit)',       r"il faut |il s'agit |il est important"),
    ('Negation (ne...pas)',                   r"n'est pas|n'a pas|ne sont pas|n'ont pas"),
    ('Reflexive (se + verb)',                 r'\bse \w+|\bs\'est \w+'),
]

for name, pattern in fr_patterns:
    examples = [s for s in qc_db1
                if re.search(pattern, s['text'], re.IGNORECASE)
                and len(s.get('concept_uris',[])) >= 2
                and not s.get('supported')
                and 40 < len(s['text']) < 250][:2]
    
    if examples:
        print(f'--- {name} ---')
        for s in examples:
            print(f"  Text: {s['text']}")
            print(f"  Entities: {[label(u) for u in s.get('concept_uris',[])[:2]]}")
            if nlp_fr:
                p = parse(s['text'], 'fr')
                print(f"  ROOT: '{p.get('root')}' (POS={p.get('pos')}) — expected meaningful verb")
            print()
        print()

=== FRENCH GRAMMATICAL STRUCTURES THAT BREAK EXTRACTION ===
Quebec sentences where spaCy ROOT is wrong — Joanie please annotate

--- Passive voice (a été + participle) ---
  Text: Au cours de la campagne électorale, il a été fortement question de la part du chef du Parti libéral et du chef du Parti québécois de la marge de manoeuvre du gouvernement.
  Entities: ['Parti libéral du Canada', 'Parti québécois']
  ROOT: 'question' (POS=NOUN) — expected meaningful verb

  Text: Il a eu comme député, de 1976 à 1985, celui qui a été premier ministre, celui qui a été le fondateur de notre parti et à qui j'ai deux bonnes raisons de vouloir rendre hommage aujourd'hui.
  Entities: ['1976', '1985']
  ROOT: 'eu' (POS=VERB) — expected meaningful verb


--- Modal + infinitive (pourrait/devrait) ---
  Text: Est-ce que le ministre pourrait nous préciser si, selon la nouvelle politique proposée par le Parti libéral, le jeune bénéficiaire d'aide sociale aura droit à l'aide sociale aux mêmes conditions qu'

In [8]:
# ── SECTION 5: SUMMARY TABLE ──────────────────────────────────────────────────
import pandas as pd

print('=== QUANTITATIVE SUMMARY ===')
rows = []
for name, db1 in [('US House', us_db1), ('Ontario LA', ont_db1), ('Quebec NA', qc_db1)]:
    total = len(db1)
    supported = sum(1 for s in db1 if s.get('supported'))
    single    = sum(1 for s in db1 if len(s.get('concept_uris',[])) == 1)
    hedged    = sum(1 for s in db1 if s.get('hedged'))
    rows.append({
        'Jurisdiction': name,
        'DB1 sentences': f'{total:,}',
        'Supported': f'{supported:,} ({supported/total*100:.1f}%)',
        'Single entity': f'{single:,} ({single/total*100:.1f}%)',
        'Hedged': f'{hedged:,} ({hedged/total*100:.1f}%)',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

=== QUANTITATIVE SUMMARY ===
Jurisdiction DB1 sentences      Supported   Single entity         Hedged
    US House       376,347 58,979 (15.7%) 200,723 (53.3%) 44,649 (11.9%)
  Ontario LA       283,033 30,115 (10.6%) 172,214 (60.8%) 31,046 (11.0%)
   Quebec NA       334,522 43,788 (13.1%) 223,987 (67.0%)     334 (0.1%)
